# Multiple Lineare Regression

**Ziel:** Sie trainieren ein Regressionsmodell für **beliebige CSV-Datensätze mit numerischer Zielvariable**. Sie müssen den Python-Code nicht vollständig verstehen: Passen Sie zunächst die **Konfigurationszelle** an, führen Sie die Codezellen von oben nach unten aus und interpretieren Sie die Ergebnisse.

**Ablauf:** CSV hochladen → Daten ansehen → Features und Zielvariable auswählen → Trainieren → Regressionsfunktion und Fehler ausgeben → Eigene Werte eingeben und vorhersagen.

**Beispiele für die Konfiguration:**
- Advertising: `FEATURES = ['TV', 'radio', 'newspaper']`, `TARGET = 'sales'`
- Fußballspieler: `FEATURES = ['goalsTotal', 'age']`, `TARGET = 'marketvalue'` (nur, wenn diese Spalten tatsächlich im CSV vorkommen).

**Hinweis:** Dieses Notebook verwendet eine **multiple lineare Regression** (ohne polynomialen Terme oder kategoriales Encoding). Die Eingabevariablen müssen numerisch sein. Sie können auch nur ein Feature wählen.

## 1. CSV-Datei hochladen

In [ ]:
# @title Ausführen (Code ausgeblendet) { display-mode: "form" }
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise ValueError('Bitte eine CSV-Datei auswählen.')

## 2. Daten einlesen und untersuchen

In [ ]:
# @title Ausführen (Code ausgeblendet) { display-mode: "form" }
import io
import pandas as pd
import numpy as np

filename = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[filename]))
# Eventuelle exportierte Indexspalten entfernen
df = df.loc[:, ~df.columns.astype(str).str.match(r'^Unnamed(:|$)')]

print('Datensatz:', filename)
print('Zeilen, Spalten:', df.shape)
print('\nSpalten:', list(df.columns))
print('\nNumerische Spalten:', df.select_dtypes(include='number').columns.tolist())
display(df.head())
display(df.describe(include='number').T.round(2))

## 3. Konfiguration – hier ändern Sie die Variablen

Wählen Sie aus den oben angezeigten Spalten **eine oder mehrere numerische Eingabevariablen** und **eine numerische Zielvariable**. Schreibweise und Groß-/Kleinschreibung müssen stimmen. Bei Bedarf `FEATURES` und `TARGET` anpassen und **alle folgenden Zellen erneut ausführen**.

In [ ]:
# ===== KONFIGURATION: NUR HIER ANPASSEN =====
FEATURES = ['TV', 'radio', 'newspaper']   # alternativ z. B. ['goalsTotal', 'age']
TARGET = 'sales'                        # alternativ z. B. 'marketvalue'
TEST_SIZE = 0.20
RANDOM_STATE = 42
# ==========================================

In [ ]:
# @title Eingaben prüfen und Daten vorbereiten { display-mode: "form" }
# Prüft Features und Zielvariable; entfernt fehlende Werte aus den ausgewählten Spalten.
if not isinstance(FEATURES, list) or not FEATURES:
    raise ValueError("FEATURES muss eine nicht-leere Liste sein, z. B. ['TV', 'radio'].")
if len(FEATURES) != len(set(FEATURES)):
    raise ValueError('Bitte kein Feature mehrfach auswählen.')
if TARGET in FEATURES:
    raise ValueError('Die Zielvariable darf nicht zugleich ein Feature sein.')
missing = [col for col in FEATURES + [TARGET] if col not in df.columns]
if missing:
    raise ValueError(f'Spalten nicht gefunden: {missing}. Vorhanden: {list(df.columns)}')
non_numeric = [col for col in FEATURES + [TARGET] if not pd.api.types.is_numeric_dtype(df[col])]
if non_numeric:
    raise ValueError(f'Diese Spalten sind nicht numerisch: {non_numeric}. Wählen Sie numerische Spalten.')

model_df = df[FEATURES + [TARGET]].replace([np.inf, -np.inf], np.nan).dropna().copy()
if len(model_df) < 10:
    raise ValueError('Zu wenige vollständige Beobachtungen (mindestens 10 erforderlich).')
print('Features:', FEATURES)
print('Zielvariable:', TARGET)
print('Verwendbare Beobachtungen:', len(model_df), '(entfernte unvollständige Zeilen:', len(df)-len(model_df), ')')

## 4. Explorative Analyse: Korrelationen und Scatterplots

Die Diagramme zeigen **jeweils einen bivariaten Zusammenhang**. Die multiple Regression betrachtet alle ausgewählten Features gemeinsam.

In [ ]:
# @title Ausführen (Code ausgeblendet) { display-mode: "form" }
import matplotlib.pyplot as plt

display(model_df.corr(numeric_only=True).round(3))
fig, axes = plt.subplots(1, len(FEATURES), figsize=(5 * len(FEATURES), 4), squeeze=False)
for ax, feature in zip(axes.ravel(), FEATURES):
    ax.scatter(model_df[feature], model_df[TARGET], alpha=0.65)
    ax.set_xlabel(feature)
    ax.set_ylabel(TARGET)
    ax.set_title(f'{TARGET} vs. {feature}')
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 5. Trainings- und Testdaten aufteilen

80 % werden standardmäßig zum Training genutzt, 20 % zur Bewertung auf bislang ungesehenen Daten.

In [ ]:
# @title Ausführen (Code ausgeblendet) { display-mode: "form" }
from sklearn.model_selection import train_test_split
X = model_df[FEATURES]
y = model_df[TARGET]
# train_test_split(): trennt Daten in Training und Test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print('Training:', len(X_train), '| Test:', len(X_test))

## 6. Multiple lineare Regression trainieren und Regressionsfunktion anzeigen

In [ ]:
# @title Ausführen (Code ausgeblendet) { display-mode: "form" }
from sklearn.linear_model import LinearRegression
model = LinearRegression()
# fit(): schätzt die Regressionskoeffizienten aus Trainingsdaten.
model.fit(X_train, y_train)

print(f'Intercept: {model.intercept_:.5f}')
print('Koeffizienten:')
display(pd.DataFrame({'Feature': FEATURES, 'Koeffizient': model.coef_}).round(5))

terms = [f'({coef:+.5f} × {feature})' for feature, coef in zip(FEATURES, model.coef_)]
print('Geschätzte Regressionsfunktion:')
print(f'{TARGET}_vorhergesagt = {model.intercept_:.5f} ' + ' '.join(terms))

## 7. Vorhersagen für Testdaten und Modellfehler

**SSE/RSS:** Summe der quadrierten Fehler; **MSE:** mittlerer quadrierter Fehler; **RMSE:** Wurzel des MSE (in der Einheit der Zielvariable).

In [ ]:
# @title Ausführen (Code ausgeblendet) { display-mode: "form" }
from sklearn.metrics import mean_squared_error

# predict(): berechnet Prognosen für bisher ungesehene Testdaten.
y_pred = model.predict(X_test)
resultate = X_test.copy()
resultate['Istwert'] = y_test.to_numpy()
resultate['Prognose'] = y_pred
resultate['Residuum'] = resultate['Istwert'] - resultate['Prognose']
display(resultate.head(10).round(3))


# MSE: mittlerer quadratischer Vorhersagefehler auf Testdaten.
mse = mean_squared_error(y_test, y_pred)
# SSE/RSS: Summe der quadrierten Residuen.
sse = float(np.sum((y_test.to_numpy() - y_pred)**2))
# RMSE: Wurzel des MSE, in der Einheit der Zielvariable.
rmse = np.sqrt(mse)
print(f'MSE      = {mse:.3f}')
print(f'RMSE     = {rmse:.3f}')
print(f'SSE/RSS  = {sse:.3f}')
print('Alle Kennzahlen wurden auf dem TEST-Datensatz berechnet.')

## 8. Tatsächliche und vorhergesagte Zielwerte vergleichen

In [ ]:
# @title Ausführen (Code ausgeblendet) { display-mode: "form" }
plt.figure(figsize=(6,5))
plt.scatter(y_test, y_pred, alpha=0.75)
lo = min(y_test.min(), y_pred.min())
hi = max(y_test.max(), y_pred.max())
plt.plot([lo,hi], [lo,hi], linestyle='--', color='black', label='Perfekte Prognose')
plt.xlabel(f'{TARGET} – tatsächlich')
plt.ylabel(f'{TARGET} – vorhergesagt')
plt.title('Tatsächlich vs. vorhergesagt')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## 9. Reflexion

1. Welche Eingabevariablen wurden ausgewählt und warum?
2. Wie lautet die geschätzte Regressionsfunktion? Wie interpretieren Sie einen Koeffizienten, **wenn die anderen Features konstant gehalten werden**?
3. Wie groß sind SSE/RSS, MSE und RMSE auf dem Testdatensatz? Welche Einheit hat der RMSE?
4. Vergleichen Sie für einzelne Testfälle **Istwert**, **Prognose** und **Residuum**. Wo liegt das Modell deutlich daneben?
5. Was ändert sich an den Testfehlern, wenn Sie ein Feature entfernen und das Notebook erneut ausführen?

**Hinweis:** Eine Korrelation beweist keine Kausalität. Nichtnumerische Merkmale müssen vor dem Training passend codiert werden; das ist nicht Bestandteil dieser Einführungsübung.